In [0]:
# ============================================================
# SHOPSTREAM — CUSTOMER CHURN PREDICTION
# Notebook : 03_churn_prediction
# Purpose  : Train churn model, track with MLflow
# Author   : El Mahdi Jamrani
# ============================================================

import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score)
from sklearn.pipeline import Pipeline

mlflow.set_registry_uri("databricks-uc")

EXPERIMENT_NAME = "/shopstream_churn_prediction"
mlflow.set_experiment(EXPERIMENT_NAME)

print("✅ MLflow setup complete.")
print(f"   Experiment: {EXPERIMENT_NAME}")
print(f"   Tracking URI: {mlflow.get_tracking_uri()}")

✅ MLflow setup complete.
   Experiment: /shopstream_churn_prediction
   Tracking URI: databricks


In [0]:
# Load the Gold customer segments table we built with dbt
df = spark.table("workspace.gold_gold.gold_customer_segments").toPandas()

print(f"Total customers loaded: {len(df):,}")
print(f"Columns: {list(df.columns)}")
print(f"\nSegment distribution:")
print(df['customer_segment'].value_counts())

Total customers loaded: 99,441
Columns: ['customer_id', 'total_orders', 'total_revenue', 'avg_order_value', 'first_order_date', 'last_order_date', 'customer_lifespan_days', 'avg_review_score', 'days_since_last_order', 'customer_segment']

Segment distribution:
customer_segment
Churned            96478
Never Purchased     2963
Name: count, dtype: int64


In [0]:
df_ml = df.copy()

# Find the last order date IN THE DATASET (not today)
# This is critical — Olist data ends in 2018
dataset_end_date = pd.to_datetime(df_ml['last_order_date']).max()
print(f"Dataset end date: {dataset_end_date.date()}")

# Recalculate days_since_last_order relative to dataset end date
df_ml['last_order_date'] = pd.to_datetime(df_ml['last_order_date'])
df_ml['days_since_last_order_relative'] = (
    dataset_end_date - df_ml['last_order_date']
).dt.days.fillna(999)

# Churn = no order in last 180 days relative to dataset end
df_ml['is_churned'] = (
    (df_ml['days_since_last_order_relative'] > 180) |
    (df_ml['customer_segment'] == 'Never Purchased')
).astype(int)

# Fill nulls
df_ml['total_orders']           = df_ml['total_orders'].fillna(0)
df_ml['total_revenue']          = df_ml['total_revenue'].fillna(0)
df_ml['avg_order_value']        = df_ml['avg_order_value'].fillna(0)
df_ml['customer_lifespan_days'] = df_ml['customer_lifespan_days'].fillna(0)
df_ml['avg_review_score']       = df_ml['avg_review_score'].fillna(0)

FEATURES = [
    'total_orders',
    'total_revenue',
    'avg_order_value',
    'customer_lifespan_days',
    'avg_review_score',
    'days_since_last_order_relative',
]

X = df_ml[FEATURES]
y = df_ml['is_churned']

print(f"\nChurn label distribution:")
print(y.value_counts())
print(f"\nChurn rate: {y.mean()*100:.1f}%")

Dataset end date: 2018-08-29

Churn label distribution:
is_churned
1    60422
0    39019
Name: count, dtype: int64

Churn rate: 60.8%


In [0]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # preserve churn ratio in both splits
)

print(f"Training set : {len(X_train):,} rows")
print(f"Test set     : {len(X_test):,} rows")
print(f"\nTrain churn rate: {y_train.mean()*100:.1f}%")
print(f"Test churn rate : {y_test.mean()*100:.1f}%")

Training set : 79,552 rows
Test set     : 19,889 rows

Train churn rate: 60.8%
Test churn rate : 60.8%


In [0]:
def evaluate_model(model, X_test, y_test):
    y_pred      = model.predict(X_test)
    y_pred_prob = model.predict_proba(X_test)[:, 1]
    return {
        "accuracy":  round(accuracy_score(y_test, y_pred), 4),
        "precision": round(precision_score(y_test, y_pred), 4),
        "recall":    round(recall_score(y_test, y_pred), 4),
        "f1":        round(f1_score(y_test, y_pred), 4),
        "roc_auc":   round(roc_auc_score(y_test, y_pred_prob), 4),
    }

# ── EXPERIMENT 1: Logistic Regression ────────────────────
with mlflow.start_run(run_name="logistic_regression"):

    params = {"C": 1.0, "max_iter": 1000, "random_state": 42}

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model",  LogisticRegression(**params))
    ])

    pipeline.fit(X_train, y_train)
    metrics = evaluate_model(pipeline, X_test, y_test)

    # Log to MLflow
    mlflow.log_params(params)
    mlflow.log_metrics(metrics)
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("features",   str(FEATURES))
    mlflow.sklearn.log_model(pipeline, "model")

    print("Logistic Regression Results:")
    for k, v in metrics.items():
        print(f"  {k:12} : {v}")

2026/06/28 13:05:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-5d69906f-499d.cloud.databricks.com/ml/experiments/1482606257106813/models/m-d22764add27e4ba4ad627dc0a03eb39d?o=7474657356822658
2026/06/28 13:05:21 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.8.1/ml/model/signatures.html for instructions on setting signature on models.


Logistic Regression Results:
  accuracy     : 0.9989
  precision    : 0.9983
  recall       : 0.9999
  f1           : 0.9991
  roc_auc      : 1.0


In [0]:
# ── EXPERIMENT 2: Random Forest ──────────────────────────
with mlflow.start_run(run_name="random_forest"):

    params = {
        "n_estimators": 100,
        "max_depth": 10,
        "random_state": 42,
        "n_jobs": -1
    }

    pipeline_rf = Pipeline([
        ("scaler", StandardScaler()),
        ("model",  RandomForestClassifier(**params))
    ])

    pipeline_rf.fit(X_train, y_train)
    metrics_rf = evaluate_model(pipeline_rf, X_test, y_test)

    # Log to MLflow
    mlflow.log_params(params)
    mlflow.log_metrics(metrics_rf)
    mlflow.log_param("model_type", "RandomForestClassifier")
    mlflow.log_param("features",   str(FEATURES))
    mlflow.sklearn.log_model(
        pipeline_rf,
        "model",
        input_example=X_test.iloc[:5]
    )

    # Feature importance
    importances = pipeline_rf.named_steps['model'].feature_importances_
    for feat, imp in sorted(zip(FEATURES, importances),
                            key=lambda x: x[1], reverse=True):
        print(f"  {feat:35} : {imp:.4f}")
        mlflow.log_metric(f"importance_{feat}", round(float(imp), 4))

    print("\nRandom Forest Results:")
    for k, v in metrics_rf.items():
        print(f"  {k:12} : {v}")

2026/06/28 13:06:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-5d69906f-499d.cloud.databricks.com/ml/experiments/1482606257106813/models/m-f8a42424929b49e09bebf998f651d622?o=7474657356822658
/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for 

  days_since_last_order_relative      : 0.9854
  avg_order_value                     : 0.0057
  total_orders                        : 0.0047
  total_revenue                       : 0.0036
  avg_review_score                    : 0.0007
  customer_lifespan_days              : 0.0000

Random Forest Results:
  accuracy     : 1.0
  precision    : 1.0
  recall       : 1.0
  f1           : 1.0
  roc_auc      : 1.0


In [0]:
# Remove days_since_last_order_relative — it's directly derived
# from the churn label definition, making it leakage not a real feature.
# In production: features must be available BEFORE the label is known.

FEATURES_CLEAN = [
    'total_orders',
    'total_revenue',
    'avg_order_value',
    'customer_lifespan_days',
    'avg_review_score',
]

X_clean = df_ml[FEATURES_CLEAN]
y_clean = df_ml['is_churned']

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X_clean, y_clean,
    test_size=0.2,
    random_state=42,
    stratify=y_clean
)

print(f"Clean features: {FEATURES_CLEAN}")
print(f"Train: {len(X_train2):,} | Test: {len(X_test2):,}")

Clean features: ['total_orders', 'total_revenue', 'avg_order_value', 'customer_lifespan_days', 'avg_review_score']
Train: 79,552 | Test: 19,889


In [0]:
# ── MODEL 1: Logistic Regression (clean) ─────────────────
with mlflow.start_run(run_name="logistic_regression_clean"):

    params_lr = {"C": 1.0, "max_iter": 1000, "random_state": 42}

    pipe_lr = Pipeline([
        ("scaler", StandardScaler()),
        ("model",  LogisticRegression(**params_lr))
    ])
    pipe_lr.fit(X_train2, y_train2)
    metrics_lr = evaluate_model(pipe_lr, X_test2, y_test2)

    mlflow.log_params(params_lr)
    mlflow.log_metrics(metrics_lr)
    mlflow.log_param("model_type", "LogisticRegression_clean")
    mlflow.log_param("features", str(FEATURES_CLEAN))
    mlflow.sklearn.log_model(pipe_lr, "model",
                             input_example=X_test2.iloc[:5])

    print("Logistic Regression (clean):")
    for k, v in metrics_lr.items():
        print(f"  {k:12} : {v}")

# ── MODEL 2: Random Forest (clean) ───────────────────────
with mlflow.start_run(run_name="random_forest_clean"):

    params_rf = {
        "n_estimators": 100,
        "max_depth": 10,
        "random_state": 42,
        "n_jobs": -1
    }

    pipe_rf = Pipeline([
        ("scaler", StandardScaler()),
        ("model",  RandomForestClassifier(**params_rf))
    ])
    pipe_rf.fit(X_train2, y_train2)
    metrics_rf = evaluate_model(pipe_rf, X_test2, y_test2)

    mlflow.log_params(params_rf)
    mlflow.log_metrics(metrics_rf)
    mlflow.log_param("model_type", "RandomForestClassifier_clean")
    mlflow.log_param("features", str(FEATURES_CLEAN))
    mlflow.sklearn.log_model(pipe_rf, "model",
                             input_example=X_test2.iloc[:5])

    importances = pipe_rf.named_steps['model'].feature_importances_
    print("\nFeature Importances (clean):")
    for feat, imp in sorted(zip(FEATURES_CLEAN, importances),
                            key=lambda x: x[1], reverse=True):
        print(f"  {feat:30} : {imp:.4f}")
        mlflow.log_metric(f"importance_{feat}", round(float(imp), 4))

    print("\nRandom Forest (clean):")
    for k, v in metrics_rf.items():
        print(f"  {k:12} : {v}")

print("\n✅ Both clean models logged to MLflow.")

2026/06/28 13:08:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-5d69906f-499d.cloud.databricks.com/ml/experiments/1482606257106813/models/m-e1d9d8b1313b4040b44b089dc297d992?o=7474657356822658
/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for 

Logistic Regression (clean):
  accuracy     : 0.6075
  precision    : 0.6077
  recall       : 0.9989
  f1           : 0.7557
  roc_auc      : 0.5512


2026/06/28 13:08:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-5d69906f-499d.cloud.databricks.com/ml/experiments/1482606257106813/models/m-e914a17f9a414284a8fc39d54f2eaf86?o=7474657356822658
/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for 


Feature Importances (clean):
  avg_order_value                : 0.4468
  total_revenue                  : 0.4038
  total_orders                   : 0.0988
  avg_review_score               : 0.0506
  customer_lifespan_days         : 0.0000

Random Forest (clean):
  accuracy     : 0.6203
  precision    : 0.617
  recall       : 0.9891
  f1           : 0.7599
  roc_auc      : 0.6397

✅ Both clean models logged to MLflow.


In [0]:
client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

# Get best run by ROC-AUC across all clean runs
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.mlflow.runName LIKE '%clean%'",
    order_by=["metrics.roc_auc DESC"],
    max_results=1
)

best_run = runs[0]
best_run_id = best_run.info.run_id

print(f"Best model   : {best_run.data.tags.get('mlflow.runName')}")
print(f"Best run ID  : {best_run_id}")
print(f"Best ROC-AUC : {best_run.data.metrics['roc_auc']}")
print(f"Best F1      : {best_run.data.metrics['f1']}")

# Register in Unity Catalog
model_uri  = f"runs:/{best_run_id}/model"
model_name = "workspace.default.shopstream_churn_model"

registered = mlflow.register_model(
    model_uri=model_uri,
    name=model_name
)

print(f"\n✅ Model registered:")
print(f"   Name   : {registered.name}")
print(f"   Version: {registered.version}")

Best model   : random_forest_clean
Best run ID  : 9a2a7bf9c6f34c0d843d48cedced0d85
Best ROC-AUC : 0.6397
Best F1      : 0.7599


Registered model 'workspace.default.shopstream_churn_model' already exists. Creating a new version of this model...
2026/06/28 13:11:21 WARNING mlflow.tracking._model_registry.fluent: Run with id 9a2a7bf9c6f34c0d843d48cedced0d85 has no artifacts at artifact path 'model', registering model based on models:/m-e914a17f9a414284a8fc39d54f2eaf86 instead


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

🔗 Created version '2' of model 'workspace.default.shopstream_churn_model': https://dbc-5d69906f-499d.cloud.databricks.com/explore/data/models/workspace/default/shopstream_churn_model/version/2?o=7474657356822658



✅ Model registered:
   Name   : workspace.default.shopstream_churn_model
   Version: 2
